Auteur:     Eva Rombouts  
Datum:      23-12-2023  
Project:    GenCareAI  
Doel:       Verkennen en opschonen van de data die met de OpenAI API is gegenereerd.

In het 010 script zijn clientprofielen gemaakt voor 24 clienten met een vorm van dementie. De profielen zijn verhalend en ongestructureerd. Dit heb ik zo gelaten omdat het in de praktijk ook vaak zo gebeurt. 
In het 015 script zijn voor deze 24 clienten 10 weken aan rapportages gegenereerd. In deze rapportages is wel een vaste structuur aangebracht, maar het blijft nog een beetje spannend of die overal goed is gevolgd en consistent is. 
In dit script gaan we de opgeslagen json file omzetten naar twee datasets: df_clienten en df_rapportages.

Note to self:
Modellen:  
MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli (multi NLI)  
wietsedv/bert-base-dutch-cased: Gebruikt door de michelin jongens. Op de site \<mask\>  
Robbert: pdelobelle/robbert-v2-dutch-base

In [4]:
# Setup
import json
import re
import pandas as pd

# Parameters
seed = 6
filename_rapportages = '../zorgdata/appelboom_rapportages.json' # Bevat de opgeslagen JSON file
filename_clienten = '../zorgdata/appelboom_clienten.json' # Hier wordt de gecleande data naar geschreven

In [5]:
# Lees de rapportages in
with open(filename_rapportages, 'r') as file:
    appelboom_rapportages = json.load(file)

De structuur van de appelboom_rapportages dictionary kan worden verbeterd en opgeschoond. 

Voor elke cliënt in appelboom_rapportages wordt een unieke ID gegenereerd in de vorm van kamer01, kamer02, enzovoorts. Deze unieke ID's vervangen de oorspronkelijke keys die beginnen met 'Naam: '.

De originele keys beginnen allemaal met 'Naam: '. Deze prefix wordt verwijderd om alleen de daadwerkelijke naam van de cliënt over te houden. Indien de naam 'Meneer' of 'Mevrouw' bevat, wordt dit gebruikt om het geslacht van de cliënt te bepalen. De naam wordt vervolgens opgeschoond door deze termen te verwijderen. Als de naam geen van deze aanduidingen bevat, wordt het geslacht als 'onbekend' gemarkeerd.

Voor elke cliënt worden de gegevens opnieuw gestructureerd. De 'Profiel' informatie wordt apart opgeslagen en alle rapportages die beginnen met 'Week' worden samengevoegd onder een aparte key 'rapportages'.

Het resultaat is een opgeschoonde en gestructureerde dictionary appelboom_clienten, waarbij elke cliënt een uniek ID heeft (het kamernummer) en de gegevens overzichtelijk zijn georganiseerd. 

In [6]:
appelboom_clienten = {}

for i, (key, value) in enumerate(appelboom_rapportages.items()):
    uniek_id = f"kamer{i+1:02d}"  # Genereert ID's zoals 'kamer01', 'kamer02', etc.
    naam = key.replace('Naam: ', '') # Verwijder prefix
    naam_geslacht = naam.split(' ', 1)
    if 'Meneer' in naam_geslacht[0]:
        geslacht = 'man' 
        naam = naam_geslacht[1]
    elif 'Mevrouw' in naam_geslacht[0]:
        geslacht = 'vrouw' 
        naam = naam_geslacht[1]
    else:
        geslacht = 'onbekend'
    
    appelboom_clienten[uniek_id] = {
        "naam": naam,
        "geslacht": geslacht,
        "profiel": value.get("Profiel", ""),
        "rapportages": {k: v for k, v in value.items() if k.startswith("Week")}
    }


In [7]:
# Maak een DataFrame voor de cliënten
df_clienten = pd.DataFrame.from_dict(appelboom_clienten, orient='index')
df_clienten.reset_index(inplace=True)
df_clienten.rename(columns={'index': 'kamernummer'}, inplace=True)
df_clienten.drop(columns=['rapportages'], inplace=True)


In [8]:
print(df_clienten)

   kamernummer                   naam  geslacht  \
0      kamer01              Pietersen       man   
1      kamer02  Sophie van der Linden  onbekend   
2      kamer03       Els van der Berg  onbekend   
3      kamer04         Hendrik de Wit       man   
4      kamer05        Alice Henderson  onbekend   
5      kamer06        Sophie de Vries     vrouw   
6      kamer07     Elisa van der Veen     vrouw   
7      kamer08         Hendrik Jansen       man   
8      kamer09          Annie Johnson  onbekend   
9      kamer10           Els de Vries     vrouw   
10     kamer11    Anna Maria de Vries  onbekend   
11     kamer12        Herman de Vries       man   
12     kamer13           Maria Jansen     vrouw   
13     kamer14          Maria de Jong     vrouw   
14     kamer15                 Janse        man   
15     kamer16        Maria De Vries      vrouw   
16     kamer17           Hannah Adams  onbekend   
17     kamer18           Elise de Wit     vrouw   
18     kamer19           Clara 

In [9]:
# # Schrijf weg. 
# with open(filename_clienten, 'w') as file:
#     json.dump(appelboom_clienten, file) 

In [10]:
# Voorbeeldje
profiel = appelboom_clienten['kamer01']['profiel']
print(profiel)

Naam: Meneer Pietersen

Dementietype: Alzheimer

Lichamelijke klachten:
- Meneer Pietersen heeft last van artritis, waardoor hij moeite heeft met bewegen en dagelijkse taken uitvoeren, zoals zichzelf aankleden en eten.
- Hij heeft ook last van gehoorverlies, wat kan leiden tot miscommunicatie en het moeilijk begrijpen van gesprekken.
- Daarnaast heeft hij een hoge bloeddruk en diabetes, waardoor er regelmatig bloeddrukcontroles en medicatiebeheer nodig zijn.

Beschrijving:
Meneer Pietersen is een vriendelijke en zachtaardige man van 78 jaar oud. Hij was vroeger bankmanager en heeft altijd een scherp geheugen gehad. Helaas begon hij een paar jaar geleden symptomen van Alzheimer te vertonen, wat leidde tot geheugenverlies en verwarring.

Vanwege zijn fysieke klachten heeft meneer Pietersen behoefte aan ondersteuning bij dagelijkse activiteiten, zoals douchen, omkleden en eten. Hij kan soms verward raken en verdwaald raken in zijn eigen vertrouwde omgeving, dus toezicht en begeleiding zij

Je ziet dat de gegenereerde data lekker biased is. Meneer Pieterse was bankmanager. Als ik snel kijk zijn er ook twee architecten, allebei man. De dames zijn artistiek, doen vrijwilligerswerk, zijn familiemensen, op waren lerares...
Maar... het is wel realistisch.

In [11]:
# voorbeeld rapportage_tekst
rapportage_tekst = appelboom_clienten['kamer01']['rapportages']['Week 1']
print(rapportage_tekst)

---StartRapportage---
Dag: Maandag
Niveau: Helpende
Rapportage: Meneer Pietersen heeft vanochtend moeite gehad met opstaan vanwege zijn artritis. Ik heb hem geholpen met aankleden en heb zijn medicijnen toegediend. Tijdens het ontbijt was hij erg verward en kon hij niet goed begrijpen wat er werd gezegd. Ik heb hem geholpen met eten en rustig tegen hem gepraat om hem te kalmeren. In de loop van de ochtend hebben we een korte wandeling gemaakt in de tuin, wat hij erg fijn vond. Onrustscore: 30
--EindRapportage---

---StartRapportage---
Dag: Dinsdag
Niveau: Verzorgende
Rapportage: Vanochtend was meneer Pietersen erg onrustig en verward. Hij wilde niet uit bed komen en was geïrriteerd tijdens het douchen. Ik heb geprobeerd hem gerust te stellen door rustig tegen hem te praten en hem te helpen met de dagelijkse activiteiten. Na het ontbijt hebben we samen wat puzzels gedaan, wat hem hielp om zijn frustratie te verminderen. Hij heeft de rest van de dag veel rust nodig gehad. Onrustscore: 60

### Parsen van weekrapportages
Bij het genereren van de data is het llm gevraagd om rapportages van een week in gestructureerde vorm te maken. 
Deze worden gestart met ---StartRapportage--- en eindigen met --EindRapportage--- (let op. Per abuis start dit met 2 streepjes). 
vervolgens zijn dag/niveau/rapportage en onrustscore vaste kopjes. De onrustscore zit vaak zonder newline aan de rapportage geplakt. 

Allereerst willen we de weekrapportages splitsen in dagrapportages

In [12]:
def split_rapportage(rapportage_tekst):
    # Combineer start- en eindpatronen in één patroon
    patroon = r'\s*-+\s*(startrapportage|eindrapportage)\s*-+\s*'
    splitpatroon = r'[\n\s]*§+[\n\s]*'

    # Vervang zowel start- als eindpatronen door §
    rapportage = re.sub(patroon, '§', rapportage_tekst, flags=re.IGNORECASE)

    # Verwijder § aan het begin en het einde (indien aanwezig)
    rapportage = re.sub(r'^' + splitpatroon, '', rapportage)
    rapportage = re.sub(splitpatroon + r'$', '', rapportage)

    # Voer een split uit
    rapportages = re.split(splitpatroon, rapportage)
    return rapportages


In [14]:
aap = split_rapportage(rapportage_tekst)
# print (rapportage_tekst)

for ap in aap:
    print('-'*60)
    print (ap)

------------------------------------------------------------
Dag: Maandag
Niveau: Helpende
Rapportage: Meneer Pietersen heeft vanochtend moeite gehad met opstaan vanwege zijn artritis. Ik heb hem geholpen met aankleden en heb zijn medicijnen toegediend. Tijdens het ontbijt was hij erg verward en kon hij niet goed begrijpen wat er werd gezegd. Ik heb hem geholpen met eten en rustig tegen hem gepraat om hem te kalmeren. In de loop van de ochtend hebben we een korte wandeling gemaakt in de tuin, wat hij erg fijn vond. Onrustscore: 30
------------------------------------------------------------
Dag: Dinsdag
Niveau: Verzorgende
Rapportage: Vanochtend was meneer Pietersen erg onrustig en verward. Hij wilde niet uit bed komen en was geïrriteerd tijdens het douchen. Ik heb geprobeerd hem gerust te stellen door rustig tegen hem te praten en hem te helpen met de dagelijkse activiteiten. Na het ontbijt hebben we samen wat puzzels gedaan, wat hem hielp om zijn frustratie te verminderen. Hij heeft 

In [35]:
def parse_dagrapportage(dagrapportage):
    # Zorg dat de onrustscore altijd op een nieuwe regel begint
    rapportage = re.sub(r'\n*(onrustscore)', '\nOnrustscore', dagrapportage, flags=re.IGNORECASE)
    rapportage_delen = re.split(r'\n', rapportage)

    parsed_data = {
        "dag": None,
        "niveau": None,
        "rapportage": None,
        "onrustscore": None
    }

    for deel in rapportage_delen:
        if deel.startswith('Dag:'):
            parsed_data["dag"] = deel[len('Dag:'):].strip()
        elif deel.startswith('Niveau:'):
            parsed_data["niveau"] = deel[len('Niveau:'):].strip()
        elif deel.startswith('Rapportage:'):
            parsed_data["rapportage"] = deel[len('Rapportage:'):].strip()
        elif deel.startswith('Onrustscore:'):
            # Zet de onrustscore om naar een integer
            score = deel[len('Onrustscore:'):].strip()
            try:
                parsed_data["onrustscore"] = int(score)
            except ValueError:
                # Handel eventuele conversiefouten af
                parsed_data["onrustscore"] = None

    return parsed_data

In [38]:
noot = parse_dagrapportage(aap[0])
for key, value in noot.items():
    print (key + ": " + str(value))

dag: Maandag
niveau: Helpende
rapportage: Meneer Pietersen heeft vanochtend moeite gehad met opstaan vanwege zijn artritis. Ik heb hem geholpen met aankleden en heb zijn medicijnen toegediend. Tijdens het ontbijt was hij erg verward en kon hij niet goed begrijpen wat er werd gezegd. Ik heb hem geholpen met eten en rustig tegen hem gepraat om hem te kalmeren. In de loop van de ochtend hebben we een korte wandeling gemaakt in de tuin, wat hij erg fijn vond.
onrustscore: 30


In [40]:
print(noot['rapportage'])

Meneer Pietersen heeft vanochtend moeite gehad met opstaan vanwege zijn artritis. Ik heb hem geholpen met aankleden en heb zijn medicijnen toegediend. Tijdens het ontbijt was hij erg verward en kon hij niet goed begrijpen wat er werd gezegd. Ik heb hem geholpen met eten en rustig tegen hem gepraat om hem te kalmeren. In de loop van de ochtend hebben we een korte wandeling gemaakt in de tuin, wat hij erg fijn vond.


In [39]:
print(aap[1])

Dag: Dinsdag
Niveau: Verzorgende
Rapportage: Vanochtend was meneer Pietersen erg onrustig en verward. Hij wilde niet uit bed komen en was geïrriteerd tijdens het douchen. Ik heb geprobeerd hem gerust te stellen door rustig tegen hem te praten en hem te helpen met de dagelijkse activiteiten. Na het ontbijt hebben we samen wat puzzels gedaan, wat hem hielp om zijn frustratie te verminderen. Hij heeft de rest van de dag veel rust nodig gehad. Onrustscore: 60


In [43]:
dagrapportages = []

for uniek_id, client_info in appelboom_clienten.items():
    for week, weekrapportage in client_info['rapportages'].items():
        dagrapportages_per_week = split_rapportage(weekrapportage)
        for dagrapportage in dagrapportages_per_week:
            rpt = parse_dagrapportage(dagrapportage)
            dagrapportages.append({
                "cliënt_id": uniek_id,
                "week": week,
                "dag": rpt['dag'],
                "niveau": rpt['niveau'],
                "rapportage": rpt['rapportage'],
                "onrustscore": rpt['onrustscore']
                # Vul hier aanvullende gegevens in, bijv. "datum", "niveau", "rapportage", "onrustscore"
            })

df_rapportages = pd.DataFrame(dagrapportages)


In [44]:
print(df_rapportages)

     cliënt_id     week        dag           niveau  \
0      kamer01   Week 1    Maandag         Helpende   
1      kamer01   Week 1    Dinsdag      Verzorgende   
2      kamer01   Week 1   Woensdag  Verpleegkundige   
3      kamer01   Week 1  Donderdag      Verzorgende   
4      kamer01   Week 1    Vrijdag         Helpende   
...        ...      ...        ...              ...   
1397   kamer24  Week 10    Dinsdag         Helpende   
1398   kamer24  Week 10   Woensdag  Verpleegkundige   
1399   kamer24  Week 10  Donderdag      Verzorgende   
1400   kamer24  Week 10    Vrijdag         Helpende   
1401   kamer24  Week 10   Zaterdag  Verpleegkundige   

                                             rapportage  onrustscore  
0     Meneer Pietersen heeft vanochtend moeite gehad...         30.0  
1     Vanochtend was meneer Pietersen erg onrustig e...         60.0  
2     Gisteravond had meneer Pietersen moeite met sl...         40.0  
3     Vandaag had meneer Pietersen veel pijn in zijn...